# Spectral Joint Flow (SJFPO) — Online Colab / Kaggle (with internet)

Clone the repo with a GitHub token, install requirements, then run training
via `!python scripts/train.py`. No source is embedded in the notebook.

## 0. Secrets and environment

1. Set `GITHUB_TOKEN` in the environment (Colab secrets or Kaggle secrets).
2. The token is read via `load_secret()` using the runtime environment.
3. Run all cells top to bottom.

In [1]:
# Detect runtime and load secrets from the right store.
import os

OUTPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/working/",
    "local": "./",
}
INPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/input/",
    "local": "./",
}
def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

ENV_NAME = detect_env()
OUTPUT = OUTPUT_DIR[ENV_NAME]

Environment: kaggle


In [2]:
# Clone the repo using the GitHub token from the environment/store.
%cd $OUTPUT
!rm -rf spectral-joint-policy
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
repo_slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/spectral-joint-policy")
repo_url = f"https://github.com/{repo_slug}.git"
if GITHUB_TOKEN:
    repo_url = repo_url.replace("github.com", f"{GITHUB_TOKEN}@github.com")
print(f"Cloning from: {repo_url.replace(GITHUB_TOKEN or '', '<TOKEN>') if GITHUB_TOKEN else repo_url}")
!git clone $repo_url spectral-joint-policy
# Move into the cloned repo and run directly (no embedded source).
%cd spectral-joint-policy
!pwd

/kaggle/working
Cloning from: https://<TOKEN>@github.com/dzungphieuluuky/spectral-joint-policy.git
Cloning into 'spectral-joint-policy'...
remote: Enumerating objects: 2939, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 2939 (delta 98), reused 152 (delta 96), pack-reused 2781 (from 1)
Receiving objects: 100% (2939/2939), 1.50 MiB | 17.43 MiB/s, done.
Resolving deltas: 100% (2007/2007), done.
/kaggle/working/spectral-joint-policy
/kaggle/working/spectral-joint-policy


In [3]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
mujoco-mjx
mujoco-warp
skrl
lerobot
jax[cuda12]
flax
jaxlib
optax
orbax-checkpoint

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus
tqdm

Overwriting requirements-colab.txt


In [4]:
# Detect runtime and choose install source.
if ENV_NAME == "colab":
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:
else:
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:

Using Python 3.12.13 environment at: /usr
Resolved 161 packages in 1.22s
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
packaging            ------------------------------     0 B/64.91 KiB
⠙ Preparing packages... (0/16)
packaging            ------------------------------ 14.78 KiB/64.91 KiB
⠙ Preparing packages... (0/16)
packaging            ------------------------------ 14.78 KiB/64.91 KiB
⠙ Preparing packages... (0/16)
packaging            ------------------------------ 14.78 KiB/64.91 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------     0 B/6.21 KiB
packaging            ------------------------------ 14.78 KiB/64.91 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
packaging            ------------------------------ 14.78 KiB/64.91 KiB
⠙ Preparing packages... (0/16)
mergedeep            ---------------------------

In [5]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [6]:
# Run training directly from the cloned repo.
!SJFPO_COMPILE=0 SJFPO_DISABLE_WANDB=0 SJFPO_MJX=0 MPLBACKEND=Agg MUJOCO_GL=egl python scripts/train.py \
    --method sjfpo \
    --env Humanoid-v5 \
    --seed 3407 \
    --total_steps 2000000 \
    --eval_steps 100 \
    --save_steps 10000

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
SJFPO policy: actor_depth=4 hidden=128 params=4,061,659
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dungngocpham171 (dungngocpham171-university-of-science) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 2t4u63ui (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/spectral-joint-policy/wandb/run-20260923_164022-2t4u63ui
wandb: Run `wandb offline` to turn off syncing.
wandb: Synci